# Language Detection — Script-Range + XLM-RoBERTa (5 languages)
### Capstone 160 — Favaz

Fixes the `langdetect` problem: `langdetect` is a character-n-gram statistical model trained on formal, native-script text -- it has no way to recognize romanized Indian languages ("nan yen madte" as Kannada), so it will always misfire on that kind of text. This notebook replaces it with a two-stage approach:

- **Case 1 -- native-script text** (Devanagari / Kannada script / Tamil script / Telugu script): a deterministic Unicode-range check. No model needed, 100% reliable for genuinely native-script text.
- **Case 2 -- Latin-script text** (English, or romanized Hindi/Kannada/Tamil/Telugu): a fine-tuned XLM-RoBERTa classifier, trained on real tweet data that already contains natural romanized/code-mixed examples (not synthetic).

**Before running anything, set up your Kaggle notebook:**
1. Right sidebar -> **Settings -> Accelerator -> GPU T4 x2** (or P100).
2. Right sidebar -> **Settings -> Internet -> On**.
3. Right sidebar -> **Add Input -> Upload -> Upload Dataset**, and upload:
   - `final_train_dataset.csv` (177,707 rows -- `text`, `label`, `lang`; `label` is unrelated leftover sentiment data, ignored here)
   - `final_test_dataset.csv` (18,540 rows, same columns)

**Output**: adds `lang` (en/hi/kn/ta/te), `is_romanized` (bool), `is_code_mixed` (bool) to each tweet -- matching the locked pipeline schema. Feeds directly into Need Detection and Sentiment Analysis (both need `lang` as input).

**A known, documented limitation** (worth stating plainly in your report): true code-mixing *within* a single tweet ("office jaana hai, traffic is crazy") is a harder problem than single-label language ID -- even AI4Bharat treats it as separate research. This notebook tags the *dominant* language per tweet and flags likely code-mixing with a lightweight heuristic (Step 3), not a full token-level model. That's a reasonable, defensible scope for a capstone.


In [1]:
# Same pinned versions as the disaster classifier notebook, for consistency
# and because they're already verified working on Kaggle's GPU.
!pip uninstall -y torchvision torchaudio -q
!pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q "transformers==4.46.3" datasets scikit-learn accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 126.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 210.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 162.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 59.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 104.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 124.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 103.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 98.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 92.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━

In [2]:
import pandas as pd
import numpy as np
import torch
import glob
import re
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)
if device.type != "cuda":
    print("WARNING: no GPU detected -- check Settings -> Accelerator, then restart the session.")

Using device: cuda


## Step 1 -- Load data and clean it up

Two small issues in the raw files, fixed here: 40 duplicate rows inside the train file, and 59 rows whose exact text appears in *both* train and test (a small leak -- if left in, the model could get credit for having already seen a test sentence during training). Both are dropped from `train_df` only; `test_df` is left completely untouched.


In [3]:
def find_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under /kaggle/input/. "
            "Make sure you added it via Add Input -> Upload -> Upload Dataset."
        )
    return matches[0]

train_df = pd.read_csv(find_file("final_train_dataset.csv"))
test_df = pd.read_csv(find_file("final_test_dataset.csv"))

for name, df in [("train", train_df), ("test", test_df)]:
    assert 'text' in df.columns and 'lang' in df.columns, \
        f"{name}_df missing expected columns. Found: {df.columns.tolist()}"

train_df = train_df.dropna(subset=['text', 'lang']).reset_index(drop=True)
test_df = test_df.dropna(subset=['text', 'lang']).reset_index(drop=True)

print("Raw shapes -- train:", train_df.shape, " test:", test_df.shape)
print("\nLanguage counts (train):\n", train_df['lang'].value_counts())

# drop exact-duplicate rows within train
before = len(train_df)
train_df = train_df.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f"\nDropped {before - len(train_df)} duplicate rows within train.")

# drop train rows whose text leaks into test
test_texts = set(test_df['text'])
before = len(train_df)
train_df = train_df[~train_df['text'].isin(test_texts)].reset_index(drop=True)
print(f"Dropped {before - len(train_df)} train rows that leaked into test.")

print("\nCleaned train shape:", train_df.shape)

Raw shapes -- train: (177707, 3)  test: (18540, 3)

Language counts (train):
 lang
hi    50000
te    50000
en    50000
ta    23881
kn     3826
Name: count, dtype: int64

Dropped 40 duplicate rows within train.
Dropped 59 train rows that leaked into test.

Cleaned train shape: (177608, 3)


## Step 2 -- Balance the training set across languages

Same problem as the disaster classifier: `en`/`hi`/`te` have 50,000 rows each, `ta` has ~23,900, but `kn` has only ~3,800 (about 2% of the data). Left as-is, the model would barely learn Kannada at all.

Fix: cap every language at 20,000 rows in training. Languages above the cap (`en`, `hi`, `te`) get randomly downsampled to 20,000 -- there's more than enough data, so this loses nothing meaningful. Languages below the cap (`kn`, and `ta` slightly) get oversampled (rows duplicated with replacement) up to 20,000. This gives a perfectly balanced 100,000-row training set and keeps runtime reasonable. `test_df` is never touched -- it stays as the original, naturally-imbalanced distribution, since that's what real-world performance should be measured against.


In [4]:
CAP_PER_LANG = 20000

balanced_frames = []
for lang, grp in train_df.groupby('lang'):
    n = len(grp)
    if n >= CAP_PER_LANG:
        balanced_frames.append(grp.sample(CAP_PER_LANG, random_state=42))
    else:
        reps = CAP_PER_LANG // n
        remainder = CAP_PER_LANG % n
        parts = [grp] * reps
        if remainder > 0:
            parts.append(grp.sample(remainder, random_state=42, replace=(remainder > n)))
        balanced_frames.append(pd.concat(parts, ignore_index=True))

train_df = pd.concat(balanced_frames, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanced train language counts:\n", train_df['lang'].value_counts())
print("\nBalanced train shape:", train_df.shape)

# carve out a small validation split for monitoring during training (stratified by lang)
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(train_df, test_size=0.05, random_state=42, stratify=train_df['lang'])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print("\nFinal train/val shapes:", train_df.shape, val_df.shape)

Balanced train language counts:
 lang
ta    20000
te    20000
en    20000
kn    20000
hi    20000
Name: count, dtype: int64

Balanced train shape: (100000, 3)

Final train/val shapes: (95000, 3) (5000, 3)


## Step 3 -- Case 1: script-range detection (no model, deterministic)

If a tweet has at least a few characters in a given language's native Unicode block, that's a completely reliable signal -- no ambiguity, no training needed. This function alone should catch every genuinely native-script Hindi/Kannada/Tamil/Telugu tweet. Everything else (English, or romanized/Latin-script Indic text) falls through to the trained classifier in Step 3 onward.

The `is_code_mixed` heuristic here is intentionally simple (see the note at the top): if a tweet has *both* meaningful native-script presence *and* a meaningful amount of Latin letters, it's flagged as likely code-mixed.


In [5]:
SCRIPT_RANGES = {
    'hi': r'[\u0900-\u097F]',   # Devanagari (Hindi)
    'kn': r'[\u0C80-\u0CFF]',   # Kannada script
    'ta': r'[\u0B80-\u0BFF]',   # Tamil script
    'te': r'[\u0C00-\u0C7F]',   # Telugu script
}

def script_char_counts(text):
    if not isinstance(text, str):
        return {}
    return {lang: len(re.findall(pattern, text)) for lang, pattern in SCRIPT_RANGES.items()}

def detect_by_script(text, min_chars=3):
    counts = script_char_counts(text)
    if not counts:
        return None
    best_lang = max(counts, key=counts.get)
    if counts[best_lang] >= min_chars:
        return best_lang
    return None

def is_latin_heavy(text, min_letters=5):
    if not isinstance(text, str):
        return False
    return len(re.findall(r'[A-Za-z]', text)) >= min_letters

def check_code_mixed(text, min_native=3, min_latin=5):
    counts = script_char_counts(text)
    has_native = counts and max(counts.values()) >= min_native
    has_latin = is_latin_heavy(text, min_latin)
    return bool(has_native and has_latin)

# Sanity check on the actual data: how much of train/test is caught by script alone
# vs. needs the trained classifier (Case 2)?
for name, df in [("train (balanced)", train_df), ("test (untouched)", test_df)]:
    caught = df['text'].apply(lambda t: detect_by_script(t) is not None)
    print(f"{name}: {caught.sum()}/{len(df)} ({100*caught.mean():.1f}%) caught by script-range alone")

train (balanced): 47876/95000 (50.4%) caught by script-range alone
test (untouched): 10753/18540 (58.0%) caught by script-range alone


## Step 4 -- Tokenize the Latin-script / ambiguous portion for the classifier

The trained classifier's job is Case 2: text where script-range detection didn't find a confident native-script match (mostly English + romanized text, but we train it on the *full* balanced set so it also learns to recognize native-script text correctly as a fallback/consistency check).


In [6]:
from transformers import XLMRobertaTokenizerFast, XLMRobertaForSequenceClassification
from datasets import Dataset

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 96  # tweets are short; keeps training faster given the larger dataset

lang2id = {lang: i for i, lang in enumerate(sorted(train_df['lang'].unique()))}
id2lang = {i: lang for lang, i in lang2id.items()}
print("Label mapping:", lang2id)

for df in (train_df, val_df, test_df):
    df['label_id'] = df['lang'].map(lang2id)

tokenizer = XLMRobertaTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_df(df):
    ds = Dataset.from_pandas(df[['text', 'label_id']].astype({'text': str}).rename(columns={'label_id': 'label'}))
    def _tok(batch):
        return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=MAX_LENGTH)
    ds = ds.map(_tok, batched=True)
    ds = ds.remove_columns(['text'])
    ds.set_format('torch')
    return ds

train_ds = tokenize_df(train_df)
val_ds = tokenize_df(val_df)
test_ds = tokenize_df(test_df)

print("Tokenized dataset sizes:", len(train_ds), len(val_ds), len(test_ds))

Label mapping: {'en': 0, 'hi': 1, 'kn': 2, 'ta': 3, 'te': 4}


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

Map:   0%|          | 0/95000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/18540 [00:00<?, ? examples/s]

Tokenized dataset sizes: 95000 5000 18540


## Step 5 -- Train

Classes are already balanced from Step 2, so no class-weighted loss is needed here (unlike the disaster classifier, which had a genuine label imbalance even after language balancing). 2 epochs given the larger dataset size (100,000 training rows) -- raise it if you have time to spare.


In [7]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import Trainer, TrainingArguments

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {'accuracy': acc, 'precision_macro': precision, 'recall_macro': recall, 'f1_macro': f1}

model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(lang2id)).to(device)

training_args = TrainingArguments(
    output_dir="/kaggle/working/langdetect_checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,0.050100,0.052017,0.985200,0.985786,0.985200,0.985273
2,0.024100,0.027770,0.994400,0.994414,0.994400,0.994399


TrainOutput(global_step=2970, training_loss=0.11705604342097786, metrics={'train_runtime': 2575.8072, 'train_samples_per_second': 73.763, 'train_steps_per_second': 1.153, 'total_flos': 9373583825280000.0, 'train_loss': 0.11705604342097786, 'epoch': 2.0})

## Step 6 -- Evaluate on the held-out (untouched, naturally-imbalanced) test set


In [8]:
predictions = trainer.predict(test_ds)
test_pred_ids = np.argmax(predictions.predictions, axis=1)
test_true_ids = test_df['label_id'].values

overall_acc = accuracy_score(test_true_ids, test_pred_ids)
p, r, f1, _ = precision_recall_fscore_support(test_true_ids, test_pred_ids, average='macro', zero_division=0)
print(f"Overall: accuracy={overall_acc*100:.2f}%  macro-precision={p:.3f}  macro-recall={r:.3f}  macro-f1={f1:.3f}")

print("\nPer-language breakdown:")
for lang, lid in sorted(lang2id.items()):
    mask = test_true_ids == lid
    if mask.sum() == 0:
        continue
    acc = accuracy_score(test_true_ids[mask], test_pred_ids[mask])
    # precision/recall for this class specifically, one-vs-rest
    p_c, r_c, f1_c, _ = precision_recall_fscore_support(
        test_true_ids == lid, test_pred_ids == lid, average='binary', zero_division=0
    )
    print(f"  {lang:4s}  n={mask.sum():5d}  accuracy={acc*100:.2f}%  precision={p_c:.3f}  recall={r_c:.3f}  f1={f1_c:.3f}")

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(test_true_ids, test_pred_ids, labels=list(range(len(lang2id))))
print("\nConfusion matrix (rows=true, cols=predicted), order:", [id2lang[i] for i in range(len(lang2id))])
print(cm)

Overall: accuracy=99.24%  macro-precision=0.979  macro-recall=0.988  macro-f1=0.983

Per-language breakdown:
  en    n= 5000  accuracy=99.20%  precision=0.991  recall=0.992  f1=0.991
  hi    n= 5000  accuracy=99.56%  precision=0.999  recall=0.996  f1=0.997
  kn    n=  497  accuracy=96.98%  precision=0.923  recall=0.970  f1=0.946
  ta    n= 3043  accuracy=98.49%  precision=0.984  recall=0.985  f1=0.985
  te    n= 5000  accuracy=99.64%  precision=0.999  recall=0.996  f1=0.998

Confusion matrix (rows=true, cols=predicted), order: ['en', 'hi', 'kn', 'ta', 'te']
[[4960    1    8   31    0]
 [  17 4978    1    1    3]
 [   1    0  482   14    0]
 [  15    0   31 2997    0]
 [  13    3    0    2 4982]]


## Step 7 -- Combined detection function (script-range first, classifier fallback)

This is the actual function that goes into the pipeline.


In [9]:
model.eval()

def predict_with_classifier(text):
    enc = tokenizer(text, max_length=MAX_LENGTH, padding='max_length', truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    pred_id = int(torch.argmax(logits, dim=1)[0])
    return id2lang[pred_id]

def detect_language_final(text):
    script_lang = detect_by_script(text)
    if script_lang is not None:
        lang = script_lang
        is_romanized = False
    else:
        lang = predict_with_classifier(text)
        is_romanized = (lang != 'en')
    is_code_mixed = check_code_mixed(text)
    return {"lang": lang, "is_romanized": is_romanized, "is_code_mixed": is_code_mixed}

print("Combined detection function ready.")

Combined detection function ready.


## Step 8 -- Sanity check on hand-picked sentences (native-script, romanized, and code-mixed)


In [10]:
sanity_tests = [
    ("this traffic is insane today", "en"),
    ("बाढ़ आ गई है, मदद चाहिए", "hi"),
    ("nan yen madthidini gottilla", "kn"),
    ("Enna teaser sema mass", "ta"),
    ("ఇది చాలా బాగుంది", "te"),
    ("office jaana hai, traffic is crazy", "hi"),   # deliberately code-mixed
]

for text, expected in sanity_tests:
    result = detect_language_final(text)
    correct = "yes" if result["lang"] == expected else "NO"
    print(f"[{correct:3s}] text={text!r:50s}  ->  {result}  (expected lang={expected})")

[yes] text='this traffic is insane today'                      ->  {'lang': 'en', 'is_romanized': False, 'is_code_mixed': False}  (expected lang=en)
[yes] text='बाढ़ आ गई है, मदद चाहिए'                           ->  {'lang': 'hi', 'is_romanized': False, 'is_code_mixed': False}  (expected lang=hi)
[yes] text='nan yen madthidini gottilla'                       ->  {'lang': 'kn', 'is_romanized': True, 'is_code_mixed': False}  (expected lang=kn)
[yes] text='Enna teaser sema mass'                             ->  {'lang': 'ta', 'is_romanized': True, 'is_code_mixed': False}  (expected lang=ta)
[yes] text='ఇది చాలా బాగుంది'                                  ->  {'lang': 'te', 'is_romanized': False, 'is_code_mixed': False}  (expected lang=te)
[NO ] text='office jaana hai, traffic is crazy'                ->  {'lang': 'en', 'is_romanized': False, 'is_code_mixed': False}  (expected lang=hi)


## Step 9 -- Run on the test set and save output matching the pipeline schema


In [11]:
output_records = []
for i in range(len(test_df)):
    row = test_df.iloc[i]
    result = detect_language_final(row['text'])
    output_records.append({
        "text": row['text'],
        "lang": result["lang"],
        "is_romanized": result["is_romanized"],
        "is_code_mixed": result["is_code_mixed"],
    })

import json
with open("/kaggle/working/language_detection_predictions.json", "w", encoding="utf-8") as f:
    json.dump(output_records, f, ensure_ascii=False, indent=2)

print("Saved /kaggle/working/language_detection_predictions.json")
print("Sample:", output_records[:3])

Saved /kaggle/working/language_detection_predictions.json
Sample: [{'text': 'Just finished working with Kicx 106 love that I get paid to listen to country music for 5 hours', 'lang': 'en', 'is_romanized': False, 'is_code_mixed': False}, {'text': 'बहुत बुरा है कि मैं आज बाहर नहीं गया. आप यह करने के लिए कुछ मजेदार की तरह लगता है', 'lang': 'hi', 'is_romanized': False, 'is_code_mixed': False}, {'text': 'yay!! मुझे सप्ताह का जिमनास्ट मिला और मुझे कभी नहीं मिला.. वह सोमवार को स्कूल नहीं गया कोई और बुरा विज्ञान शिक्षक नहीं हैय्याय्याय्याय्याय ओह नहीं मेरे पास परीक्षाएं हैं', 'lang': 'hi', 'is_romanized': False, 'is_code_mixed': False}]


## Step 10 -- Save the model, tokenizer, and label mapping


In [12]:
model.save_pretrained("/kaggle/working/language_detection_xlmr")
tokenizer.save_pretrained("/kaggle/working/language_detection_xlmr")

import json
with open("/kaggle/working/lang2id.json", "w", encoding="utf-8") as f:
    json.dump(lang2id, f, ensure_ascii=False, indent=2)

print("Saved model + tokenizer to /kaggle/working/language_detection_xlmr")
print("Saved label mapping to /kaggle/working/lang2id.json")
print("\nDownload (from Output panel): the language_detection_xlmr folder (all files inside),")
print("plus lang2id.json and language_detection_predictions.json -- same pattern as the disaster classifier.")

Saved model + tokenizer to /kaggle/working/language_detection_xlmr
Saved label mapping to /kaggle/working/lang2id.json

Download (from Output panel): the language_detection_xlmr folder (all files inside),
plus lang2id.json and language_detection_predictions.json -- same pattern as the disaster classifier.


In [13]:
import shutil
shutil.make_archive("/kaggle/working/language_detection_xlmr", "zip", "/kaggle/working/language_detection_xlmr")
print("Zipped -> /kaggle/working/language_detection_xlmr.zip")

Zipped -> /kaggle/working/language_detection_xlmr.zip


## What's next

Feed `detect_language_final()`'s output (`lang`, `is_romanized`, `is_code_mixed`) into Need Detection and Sentiment Analysis -- both run in parallel and both need `lang` as an input, per the locked pipeline. Need Detection is the next piece to retrain on the same 5-language expanded dataset the disaster classifier used.
